# Linear combination of input series

A factor model on log returns, column by column: `r[t] = constant * dt[t] + sum_i weights[i] * r_i[t] + sigma * sqrt(dt[t]) * eps[t]`. The inputs are log returns (simulated here; for real prices use `fm.from_prices`) and the output is log returns with exactly their index and columns. `constant` is an annualised drift. The output has no natural starting price, so `fm.to_prices` needs `initial_price`.

Edit the parameters cell and re-run all: the same parameters and seeds always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fantasma as fm

## Parameters

Weights, constant and sigma are each a scalar (all columns), a list with one value per column, or a Series keyed by column. `constant` (annualised drift) and `sigma` (annualised volatility) enter each row as `constant * dt` and `sigma * sqrt(dt) * eps`, with `dt` the time since the previous row in years.

In [ ]:
# Input series: N_INPUTS log-return frames with the same index and columns. Here they are
# simulated random walks, one column per asset; real prices work too once converted, e.g. from
# Almanac: [fm.from_prices(load(...)["close"].unstack("symbol")), ...].
INPUT_SEED = 1
N_INPUTS = 2
ASSETS = ["X", "Y"]
INPUT_SIGMA = 0.2
START = "2024-01-01"
FREQ = "1d"
N_STEPS = 365 * 3

# Simulation
SEED = 42
N_PATHS = 3

# Model: one weight per input series
WEIGHTS = [1.0, [0.5, -0.5]]
CONSTANT = 0.1  # annualised drift
INITIAL_PRICE = 100.0  # starting price of the output, for plotting
SIGMA = 0.05


## Inputs

In [ ]:
input_model = fm.Composite({a: fm.RandomWalk(INPUT_SIGMA, asset=a) for a in ASSETS})
inputs = fm.simulate(
    input_model, start=START, freq=FREQ, n_steps=N_STEPS, seed=INPUT_SEED, n_paths=N_INPUTS
)  # log returns
inputs[0].head()

## Simulate

In [ ]:
model = fm.LinearCombination(WEIGHTS, constant=CONSTANT, sigma=SIGMA)
model

In [ ]:
paths = fm.simulate_driven(model, inputs, seed=SEED, n_paths=N_PATHS)
out = paths[0]
out.head()

In [ ]:
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(10, 3 * len(ASSETS)), sharex=True)
for ax, asset in zip(np.atleast_1d(axes), ASSETS):
    for i, x in enumerate(inputs):
        px = fm.to_prices(x)
        ax.plot(px.index, px[asset], lw=0.8, alpha=0.6, label=f"input {i}")
    for k, p in enumerate(paths):
        px = fm.to_prices(p, initial_price=INITIAL_PRICE)
        ax.plot(px.index, px[asset], lw=0.8, color="k", alpha=0.5 if k else 1, label="output" if k == 0 else None)
    ax.set_ylabel("price")
    ax.set_title(asset)
    ax.legend(fontsize=8)
plt.show()

## Check: regression recovers the parameters

OLS of each output column on the input columns, with a constant: the intercept over `dt` estimates the annualised drift `constant` (noisy unless the sample is long), and the residual standard deviation over `sqrt(dt)` estimates `sigma`.

In [ ]:
dt = (out.index[1] - out.index[0]) / pd.Timedelta(days=365.25)
W, c, s = model.resolve(out.columns)
for j, asset in enumerate(out.columns):
    X = np.column_stack([np.ones(len(out))] + [x[asset].to_numpy() for x in inputs])
    beta, *_ = np.linalg.lstsq(X, out[asset].to_numpy(), rcond=None)
    resid = out[asset].to_numpy() - X @ beta
    sigma_hat = resid.std(ddof=X.shape[1]) / np.sqrt(dt)
    print(f"{asset}: constant {c[j]:+.3f} -> {beta[0] / dt:+.3f}, "
          f"weights {np.round(W[:, j], 3)} -> {np.round(beta[1:], 3)}, sigma {s[j]:.3f} -> {sigma_hat:.3f}")